# 02 · Model comparison (5-fold cross-validation by person)

| group | models |
|---|---|
| classical baselines | logistic regression, random forest, XGBoost (on the 46-number summary row + tempo) |
| neural models (attention pooling → α) | 1D-CNN, BiLSTM, BiGRU, CNN-LSTM, TCN, ST-Transformer |

Every neural model ends with **attention pooling over the 64 frames**: its weights α say which moments decided the class.
The final product uses α to light up the error region on the 3D body at the right time, so α is checked too.

**Evaluation protocol (identical for every model and every variant).**
* **5-fold cross-validation grouped by person** (StratifiedGroupKFold, k = 5): each fold tests 3-4 people the model has
  never seen; every person is tested once per seed.
* **3 seeds** (0, 1, 2): 15 runs per model; the reported number is the mean ± standard deviation over the 15 runs.
* **Validation persons**: inside each fold, 2 training persons choose the best epoch;
  the test persons are never used for any choice.
* **Early stopping**: training stops when the validation macro-F1 has not improved for **20 epochs** (at most 60 epochs);
  the weights of the best epoch are kept and saved as a checkpoint (`<model>/fold_<seed>_<k>/best.pt`, with the best epoch,
  its validation F1, the number of epochs run, the normalisation statistics and the test metrics).
* **Normalisation** statistics from the training part of each fold only; **loss weights** balance the classes and let
  every recording count equally.
* **Metrics**: macro-F1 per rep (main), accuracy, macro-F1 per recording (mean probability over its reps), per-class F1,
  confusion matrix; **paired Wilcoxon test** on the 15 identical runs when two models or variants are compared.

**How to run.** Run All after notebook 01. About 1 hour on the laptop RTX 3050 (the ST-Transformer takes ~50 min of it).
The monitor cell shows progress and ETA; interrupting it does not stop the training.

**Outputs.** `results/02_model_comparison/` (comparison tables, figures, one report per model, `best_model.json`).

In [1]:
# ---- setup (every notebook starts with this cell) ----
import sys, json, time, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()     # research_folders/
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output, Image
pd.set_option("display.precision", 3); pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 120)
from squat_research.common import load_config, apply_overrides, results_dir, nice, read_json, write_json
from squat_research.dataset import build_dataset, load_dataset
from squat_research.engine import comparison_table, select_best, fold_scores, paired_test
from squat_research import viz, monitor
cfg = load_config()
print("research folder:", ROOT)
from squat_research.models import MODEL_INFO, build_model, n_params, TREE_MODELS
from squat_research.report import model_report
OUT = results_dir(cfg, "02_model_comparison"); FIG = results_dir(cfg, "02_model_comparison", "figures")
REP = results_dir(cfg, "02_model_comparison", "model_reports"); RUN = OUT / "cv"
ds = load_dataset(cfg); C = ds["classes"]
MODELS = cfg["baseline_models"] + cfg["neural_models"]

## 1. The models: what each receives and how it processes it

In [2]:
T, J, F = ds["joints"].shape[1], ds["joints"].shape[2], ds["frame"].shape[2]
models_tab = pd.DataFrame([{"model": nice(m), "family": MODEL_INFO[m]["family"], "input": MODEL_INFO[m]["input"],
                            "processing": MODEL_INFO[m]["processing"],
                            "parameters": "-" if m in TREE_MODELS else f"{n_params(build_model(m, J, F, T, len(C), ds['meta']['joint_names'], cfg['model'])):,}"}
                           for m in MODELS])
models_tab.to_csv(OUT / "models_overview.csv", index=False); display(models_tab)

## 2. Train all models (background process + live monitor)

In [3]:
if not (RUN / "progress.json").exists() or read_json(RUN / "progress.json")["state"] != "done":
    proc = monitor.launch(MODELS, RUN, save_models=True)     # checkpoints: <model>/fold_<seed>_<k>/best.pt (best epoch)
monitor.watch(RUN, every=30)      # interrupt to stop watching; the training continues (monitor.stop(RUN) stops it)

## 3. Results

In [4]:
table = comparison_table(RUN)
table["group"] = ["classical baseline" if m in TREE_MODELS else "neural" for m in table["model"]]
table.to_csv(OUT / "comparison.csv", index=False)
(OUT / "comparison.md").write_text(table.to_markdown(index=False, floatfmt=".3f"), encoding="utf-8")
display(table[["name", "group", "macro_f1", "macro_f1_std", "accuracy", "rec_macro_f1", "alpha_drop", "random_drop", "params", "median_best_epoch", "minutes"]])

In [5]:
viz.comparison(table, FIG, title="Macro-F1 per rep (5-fold cross-validation by person x 3 seeds)"); plt.show()

In [6]:
viz.fold_spread(RUN, list(table["model"]), FIG); plt.show()

In [7]:
viz.per_class(RUN, list(table["model"]), C, FIG); plt.show()

In [8]:
viz.confusions(RUN, list(table["model"]), C, FIG); plt.show()

In [9]:
viz.person_accuracy(RUN, list(table["model"]), FIG); plt.show()

## 4. Does α point at the frames that matter?

For every test rep, the 10 % of frames with the highest α are replaced by the rep's mean and the drop of the true-class
probability is measured; the same with 10 % random frames (3 repeats). A meaningful α hurts clearly more when masked.

In [10]:
viz.deletion(table[table["alpha_drop"].notna()], FIG); plt.show()

## 5. One report per model (input processing and performance)

In [11]:
for m in table["model"]:
    display(Markdown(model_report(RUN, m, ds, cfg, REP)))
    for f in ["confusion.png", f"learning_curves_{m}.png", f"alpha_profile_{m}.png"]:
        if (REP / m / f).exists():
            display(Image(filename=str(REP / m / f), width=640))

## 6. Choosing the best model

**Rule** (fixed before looking at the numbers):
1. Candidates: neural models whose α passes the check (masking the α frames drops the true-class probability ≥ 2 × more than random frames).
2. Highest mean macro-F1 over the 15 runs; models within one standard deviation of it count as tied.
3. Among tied models: higher recording-level macro-F1, then fewer parameters (the product runs on a CPU).

The paired test compares the chosen model with every other model on the same 15 runs (p < 0.05 = a real difference).

In [12]:
neural = table[table["group"] == "neural"].reset_index(drop=True)
best, rule = select_best(neural)
display(rule[["name", "macro_f1", "macro_f1_std", "rec_macro_f1", "alpha_drop", "random_drop", "alpha_ok", "tied_with_best", "params", "chosen"]])
fb = fold_scores(RUN, best)
paired = pd.DataFrame([{"compared with": r["name"], **paired_test(fb, fold_scores(RUN, r["model"]))} for _, r in table.iterrows() if r["model"] != best])
paired.to_csv(OUT / "paired_tests_best_vs_others.csv", index=False); display(paired)

In [13]:
b = table.set_index("model").loc[best]
info = dict(model=best, name=nice(best), macro_f1=b["macro_f1"], macro_f1_std=b["macro_f1_std"], rec_macro_f1=b["rec_macro_f1"],
            alpha_drop=b["alpha_drop"], random_drop=b["random_drop"], params=int(b["params"]), median_best_epoch=b["median_best_epoch"],
            tied=list(rule.loc[rule["tied_with_best"], "model"]), minutes_per_15_runs=b["minutes"])
write_json(OUT / "best_model.json", info)
cls = table[table["group"] == "classical baseline"].iloc[0]
txt = (f"**Chosen model: {info['name']}**: macro-F1 {info['macro_f1']:.3f} ± {info['macro_f1_std']:.3f}, recording level "
       f"{info['rec_macro_f1']:.3f}, α check {info['alpha_drop']:.3f} vs random {info['random_drop']:.3f}, {info['params']:,} parameters. "
       f"Tied within one standard deviation: {', '.join(nice(m) for m in info['tied'])}. Best classical baseline: {cls['name']} ({cls['macro_f1']:.3f}).")
(OUT / "summary.md").write_text(txt, encoding="utf-8"); display(Markdown(txt))